In [1]:
import pandas as pd

from results_notebook_setup import results_loader, significant_models

100%|██████████| 2/2 [00:01<00:00,  1.51it/s]


In [2]:
short_code_result = results_loader.short_code
short_code_result_clean = short_code_result.get_clean_data_object()

long_code_result = results_loader.long_code
long_code_result_clean = long_code_result.get_clean_data_object()


100%|██████████| 2/2 [00:00<00:00,  2.50it/s]


In [3]:
vek = {'model_order': significant_models}

In [4]:
def match_index(s, df):
    if not isinstance(df.index, pd.MultiIndex) or df.index.nlevels < 2:
        return s

    if df.index.nlevels > 2:
        raise RuntimeError('Matching dataframe index for a multi-index of more than 2 levels not implemented')

    index_names = df.index.names
    s_matched = pd.DataFrame({
        k: s for k in df.index.get_level_values(index_names[1]).unique()
    }).stack()
    s_matched.index.names = index_names
    return s_matched


def print_comparison_summary(comparison_df):
    print(f"\nAgreement rate: {comparison_df.agreement.sum()}/{len(comparison_df)}")

    if (~comparison_df.agreement).sum():
        print(f"Disagreement cases:\n{comparison_df[~comparison_df.agreement][
            ['original_p', 'clean_p', 'original_failed', 'clean_failed']]}")

    er = comparison_df.exclusion_rate
    print(f"\nExclusion rate max / mean: {er.max():.3f} / {er.mean():.3f}\n")


def compare(orig_res, clean_res, alpha=0.05, effect='variant', model_order: list[str] | None = None):
    (orig_df, orig_sum), (clean_df, clean_sum) = [getattr(res, f'{effect}_effect') for res in (orig_res, clean_res)]
    comparison_df = pd.DataFrame({
        'original_p': orig_df.p_value,
        'clean_p': clean_df.p_value
    })
    comparison_df['original_significant'] = comparison_df['original_p'] < alpha
    comparison_df['clean_significant'] = comparison_df['clean_p'] < alpha
    comparison_df['agreement'] = comparison_df['original_significant'] == comparison_df['clean_significant']

    comparison_df['estimate_diff'] = clean_df['estimate'] - orig_df['estimate']

    n_resp, n_clean_resp = [res.mres.variants['main'].full_data.groupby('model').size() for res in (orig_res, clean_res)]
    er = 1 - n_clean_resp / n_resp
    comparison_df['exclusion_rate'] = match_index(er, comparison_df)

    if model_order is not None:
        comparison_df.sort_index(
            level='model',
            key=lambda idx: idx.map({model: i for i, model in enumerate(model_order)}),
            inplace=True
        )

    fi_orig, fi_clean = [
        (
                sum_df.fit_failed | sum_df.is_singular | sum_df.convergence_messages
        ) for sum_df in (orig_sum, clean_sum)
    ]

    comparison_df['original_failed'] = match_index(fi_orig, comparison_df)
    comparison_df['clean_failed'] = match_index(fi_clean, comparison_df)

    print_comparison_summary(comparison_df)

    return comparison_df


In [5]:
compare(short_code_result, short_code_result_clean, **vek)



Agreement rate: 9/10
Disagreement cases:
             original_p   clean_p  original_failed  clean_failed
model                                                           
gemma-7b-it     0.03235  0.051153            False         False

Exclusion rate max / mean: 0.166 / 0.059



,original_p,clean_p,original_significant,clean_significant,agreement,estimate_diff,exclusion_rate,original_failed,clean_failed
model,,,,,,,,,
phi-2,0.174981,0.124210,False,False,True,0.061916,0.0084,False,False
Phi-3.5-mini-instruct,0.093131,0.108751,False,False,True,0.027460,0.0048,False,False
gemma-2b,0.838333,0.933579,False,False,True,0.043993,0.0952,False,False
gemma-2-9b,0.446395,0.482820,False,False,True,0.018577,0.0170,False,False
Mathstral-7B-v0.1,0.138369,0.159018,False,False,True,-0.013820,0.0138,False,False
Meta-Llama-3-8B-Instruct,0.115889,0.601477,False,False,True,0.287278,0.1660,False,False
gemma-7b-it,0.032350,0.051153,True,False,False,0.027299,0.0728,False,False
Meta-Llama-3-8B,0.025193,0.007575,True,True,True,0.147413,0.0538,False,False
Mistral-7B-Instruct-v0.1,0.470808,0.400407,False,False,True,0.051506,0.0654,False,False


In [6]:
compare(long_code_result, long_code_result_clean, **vek)

gemma-2-2b - convergence messages: Model failed to converge with max|grad| = 0.162445 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting.; Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?
gemma-2-2b - convergence messages: Model failed to converge with max|grad| = 0.162169 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting.; Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?



Agreement rate: 10/10

Exclusion rate max / mean: 0.084 / 0.042



,original_p,clean_p,original_significant,clean_significant,agreement,estimate_diff,exclusion_rate,original_failed,clean_failed
model,,,,,,,,,
phi-2,0.430425,0.354964,False,False,True,-0.053205,0.0048,False,False
Phi-3.5-mini-instruct,0.257786,0.291356,False,False,True,0.031570,0.0046,False,False
gemma-2b,0.320604,0.390951,False,False,True,0.038970,0.0836,False,False
gemma-2-9b,0.975788,0.873162,False,False,True,-0.049359,0.0168,False,False
Mathstral-7B-v0.1,0.529166,0.413614,False,False,True,0.079406,0.0216,False,False
Meta-Llama-3-8B-Instruct,0.897821,0.930887,False,False,True,0.073447,0.0322,False,False
gemma-7b-it,0.415899,0.346681,False,False,True,-0.057162,0.0744,False,False
Meta-Llama-3-8B,0.771372,0.602966,False,False,True,0.073834,0.0328,False,False
Mistral-7B-Instruct-v0.1,0.371323,0.291438,False,False,True,-0.063586,0.0784,False,False


In [7]:
compare(short_code_result, short_code_result_clean, effect='number', **vek)

phi-2 - convergence messages: Model failed to converge with max|grad| = 0.190118 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting.; Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?



Agreement rate: 17/20
Disagreement cases:
                            original_p   clean_p  original_failed  \
model           variable                                            
phi-2           is_variant    0.125491  0.000000            False   
                sum_logs_c    0.277163  0.000000            False   
Meta-Llama-3-8B is_variant    0.071158  0.018158            False   

                            clean_failed  
model           variable                  
phi-2           is_variant          True  
                sum_logs_c          True  
Meta-Llama-3-8B is_variant         False  

Exclusion rate max / mean: 0.166 / 0.059



original_p   clean_p  \
model                    variable                           
phi-2                    is_variant    0.125491  0.000000   
                         sum_logs_c    0.277163  0.000000   
Phi-3.5-mini-instruct    is_variant    0.084888  0.098840   
                         sum_logs_c    0.714063  0.712506   
gemma-2b                 is_variant    0.910404  0.996139   
                         sum_logs_c    0.747096  0.746487   
gemma-2-9b               is_variant    0.227705  0.265635   
                         sum_logs_c    0.083257  0.116496   
Mathstral-7B-v0.1        is_variant    0.103195  0.113658   
                         sum_logs_c    0.176232  0.149221   
Meta-Llama-3-8B-Instruct is_variant    0.119200  0.702969   
                         sum_logs_c    0.865857  0.704794   
gemma-7b-it              is_variant    0.127323  0.137114   
                         sum_logs_c    0.055340  0.160083   
Meta-Llama-3-8B          is_variant    0.071158  0.018158   
                         sum_logs_c    0.120908  0.354708   
Mistral-7B-Instruct-v0.1 is_variant    0.575829  0.453859   
                         sum_logs_c    0.654940  0.831550   
gemma-2-2b               is_variant    0.610790  0.619039   
                         sum_logs_c    0.189736  0.195669   

                                     original_significant  clean_significant  \
model                    variable                                              
phi-2                    is_variant                 False               True   
                         sum_logs_c                 False               True   
Phi-3.5-mini-instruct    is_variant                 False              False   
                         sum_logs_c                 False              False   
gemma-2b                 is_variant                 False              False   
                         sum_logs_c                 False              False   
gemma-2-9b               is_variant                 False              False   
                         sum_logs_c                 False              False   
Mathstral-7B-v0.1        is_variant                 False              False   
                         sum_logs_c                 False              False   
Meta-Llama-3-8B-Instruct is_variant                 False              False   
                         sum_logs_c                 False              False   
gemma-7b-it              is_variant                 False              False   
                         sum_logs_c                 False              False   
Meta-Llama-3-8B          is_variant                 False               True   
                         sum_logs_c                 False              False   
Mistral-7B-Instruct-v0.1 is_variant                 False              False   
                         sum_logs_c                 False              False   
gemma-2-2b               is_variant                 False              False   
                         sum_logs_c                 False              False   

                                     agreement  estimate_diff  exclusion_rate  \
model                    variable                                               
phi-2                    is_variant      False       0.094158          0.0084   
                         sum_logs_c      False      -0.033653          0.0084   
Phi-3.5-mini-instruct    is_variant       True       0.026993          0.0048   
                         sum_logs_c       True       0.000598          0.0048   
gemma-2b                 is_variant       True       0.044941          0.0952   
                         sum_logs_c       True      -0.000332          0.0952   
gemma-2-9b               is_variant       True       0.029991          0.0170   
                         sum_logs_c       True      -0.013072          0.0170   
Mathstral-7B-v0.1        is_variant       True      -0.002251          0.0138   
                         sum_logs_c       

In [8]:
compare(long_code_result, long_code_result_clean, effect='number', **vek)


gemma-2-2b - convergence messages: Model failed to converge with max|grad| = 0.161462 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting.; Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?
gemma-2-2b - convergence messages: Model failed to converge with max|grad| = 0.161092 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting.; Model is nearly unidentifiable: very large eigenvalue
 - Rescale variables?



Agreement rate: 20/20

Exclusion rate max / mean: 0.084 / 0.042



original_p       clean_p  \
model                    variable                               
phi-2                    is_variant    0.433329  3.607134e-01   
                         sum_logs_c    0.971648  9.789136e-01   
Phi-3.5-mini-instruct    is_variant    0.480131  5.258021e-01   
                         sum_logs_c    0.171258  1.838934e-01   
gemma-2b                 is_variant    0.388527  4.962791e-01   
                         sum_logs_c    0.715809  5.802897e-01   
gemma-2-9b               is_variant    0.990047  9.772730e-01   
                         sum_logs_c    0.407782  3.915580e-01   
Mathstral-7B-v0.1        is_variant    0.618671  5.813467e-01   
                         sum_logs_c    0.526755  8.540730e-01   
Meta-Llama-3-8B-Instruct is_variant    0.535686  7.494367e-01   
                         sum_logs_c    0.373097  6.300693e-01   
gemma-7b-it              is_variant    0.766857  6.191222e-01   
                         sum_logs_c    0.082139  1.625356e-01   
Meta-Llama-3-8B          is_variant    0.698976  5.481767e-01   
                         sum_logs_c    0.626791  6.447067e-01   
Mistral-7B-Instruct-v0.1 is_variant    0.316342  2.668899e-01   
                         sum_logs_c    0.625269  7.604736e-01   
gemma-2-2b               is_variant    0.000000  0.000000e+00   
                         sum_logs_c    0.000000  1.134556e-18   

                                     original_significant  clean_significant  \
model                    variable                                              
phi-2                    is_variant                 False              False   
                         sum_logs_c                 False              False   
Phi-3.5-mini-instruct    is_variant                 False              False   
                         sum_logs_c                 False              False   
gemma-2b                 is_variant                 False              False   
                         sum_logs_c                 False              False   
gemma-2-9b               is_variant                 False              False   
                         sum_logs_c                 False              False   
Mathstral-7B-v0.1        is_variant                 False              False   
                         sum_logs_c                 False              False   
Meta-Llama-3-8B-Instruct is_variant                 False              False   
                         sum_logs_c                 False              False   
gemma-7b-it              is_variant                 False              False   
                         sum_logs_c                 False              False   
Meta-Llama-3-8B          is_variant                 False              False   
                         sum_logs_c                 False              False   
Mistral-7B-Instruct-v0.1 is_variant                 False              False   
                         sum_logs_c                 False              False   
gemma-2-2b               is_variant                  True               True   
                         sum_logs_c                  True               True   

                                     agreement  estimate_diff  exclusion_rate  \
model                    variable                                               
phi-2                    is_variant       True      -0.052860          0.0048   
                         sum_logs_c       True      -0.000803          0.0048   
Phi-3.5-mini-instruct    is_variant       True       0.029915          0.0046   
                         sum_logs_c       True       0.001757          0.0046   
gemma-2b                 is_variant       True       0.060499          0.0836   
                         sum_logs_c       True      -0.016686          0.0836   
gemma-2-9b               is_variant       True       0.006316          0.0168   
                         sum_logs_c       True      -0.003714          0.0168   
Mathstral-7B-v0.1        is_variant